# Airbnb Open Data — EDA Project
Consolidated notebook: data loading, structural checks, and outlier
investigation completed so far. Every cell below has a comment
explaining *why* the step exists, not just what it does — you should
be able to explain each one without looking at the code.


## 1. Load the data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Raw string (r"...") is required on Windows paths with backslashes,
# otherwise sequences like \U get misread as escape characters.
path = r"C:\Users\yashv\OneDrive\Desktop\PROJECTS\air_bn\Airbnb_Open_Data.csv"
df = pd.read_csv(path)

print(df.shape)
df.head()


(102599, 26)


C:\Users\yashv\AppData\Local\Temp\ipykernel_9356\1150889372.py:8: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN
3,1002755,NaN,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,$74,30.0,270.0,7/5/2019,4.64,4.0,1.0,322.0,NaN,NaN
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,$41,10.0,9.0,11/19/2018,0.10,3.0,1.0,289.0,"Please no smoking in the house, porch or on th...",NaN


## 2. Standardize column names
Source data mixes `snake_case`, `Title Case`, and spaced names
(e.g. `"host id"`, `NAME`, `"neighbourhood group"`). Fixing this once,
up front, avoids KeyErrors from typos/case mismatches for the rest
of the notebook.

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
print(df.columns.tolist())


['id', 'name', 'host_id', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'construction_year', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365', 'house_rules', 'license']


## 3. Structural check — shape, dtypes, nulls
Always run this before touching anything else. It tells you which
columns are usable as-is, which need type conversion, and which are
too sparse to build core questions on.

In [4]:
print(df.shape)
print(df.info())
print(df.isnull().sum())


(102599, 26)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   name                            102349 non-null  object 
 2   host_id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host_name                       102193 non-null  object 
 5   neighbourhood_group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country_code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  can

In [5]:
print(df.isnull().sum())

id                                     0
name                                 250
host_id                                0
host_identity_verified               289
host_name                            406
neighbourhood_group                   29
neighbourhood                         16
lat                                    8
long                                   8
country                              532
country_code                         131
instant_bookable                     105
cancellation_policy                   76
room_type                              0
construction_year                    214
price                                247
service_fee                          273
minimum_nights                       409
number_of_reviews                    183
last_review                        15893
reviews_per_month                  15879
review_rate_number                   326
calculated_host_listings_count       319
availability_365                     448
house_rules     

### Findings from the null check
- `license`: ~99.6% missing → drop the column entirely, not worth imputing.
- `house_rules`: ~50.6% missing → too sparse for core analysis; optional
  text-mining stretch question only.
- `last_review` / `reviews_per_month`: ~15,800 missing each, almost
  identical counts → hypothesis: these are listings that have never
  been reviewed, not random missingness. Tested below.
- Everything else (`neighbourhood_group`, `room_type`, `price`,
  `instant_bookable`, `cancellation_policy`, `review_rate_number`)
  sits under 1% missing — safe to drop those few rows per-question
  rather than needing a special strategy.

## 4. Test the "never reviewed" hypothesis
If `last_review` is null because a listing has zero reviews, then
`number_of_reviews` for those same rows should cluster near 0.

In [6]:
never_reviewed_check = df[df['last_review'].isnull()]['number_of_reviews'].describe()
print(never_reviewed_check)


count    15769.000000
mean         0.159110
std          4.841932
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max        228.000000
Name: number_of_reviews, dtype: float64


**Result:** 75% of the missing-`last_review` rows have exactly 0
reviews — confirms the hypothesis for most rows. BUT mean is ~0.16
and max is 228, meaning a small tail of listings has real reviews
but missing review-timing metadata. That's a genuine data quality
gap, not "no reviews yet."

**Decision:** don't blanket-fill `reviews_per_month` with 0 across
all nulls. Split the treatment:
- 0-review rows → fill `reviews_per_month` with 0 (accurate).
- non-zero-review rows with null timing → flag separately, note as
  a known data quality issue, don't silently impute.

## 5. Outlier checks — `minimum_nights` and `availability_365`
Both columns have physically impossible values (negative nights,
availability over 365 days/year). Outliers like this silently
distort any mean/groupby without ever showing up in a null check —
they have to be checked explicitly.

In [7]:
print(df.describe())

                 id       host_id            lat           long  \
count  1.025990e+05  1.025990e+05  102591.000000  102591.000000   
mean   2.914623e+07  4.925411e+10      40.728094     -73.949644   
std    1.625751e+07  2.853900e+10       0.055857       0.049521   
min    1.001254e+06  1.236005e+08      40.499790     -74.249840   
25%    1.508581e+07  2.458333e+10      40.688740     -73.982580   
50%    2.913660e+07  4.911774e+10      40.722290     -73.954440   
75%    4.320120e+07  7.399650e+10      40.762760     -73.932350   
max    5.736742e+07  9.876313e+10      40.916970     -73.705220   

       construction_year  minimum_nights  number_of_reviews  \
count      102385.000000   102190.000000      102416.000000   
mean         2012.487464        8.135845          27.483743   
std             5.765556       30.553781          49.508954   
min          2003.000000    -1223.000000           0.000000   
25%          2007.000000        2.000000           1.000000   
50%          2012.

In [8]:
# minimum_nights < 0 — impossible, can't require a negative stay
neg_min_nights = df[df['minimum_nights'] < 0]
print("minimum_nights < 0:", len(neg_min_nights))


minimum_nights < 0: 13


In [9]:
# minimum_nights > 365 — implausible for a short-term rental
high_min_nights = df[df['minimum_nights'] > 365]
print("minimum_nights > 365:", len(high_min_nights))
print(high_min_nights[['room_type', 'minimum_nights', 'availability_365']].describe())


minimum_nights > 365: 35
       minimum_nights  availability_365
count        35.00000         33.000000
mean        882.80000        197.242424
std        1045.96777        124.938492
min         366.00000          0.000000
25%         400.00000         90.000000
50%         500.00000        193.000000
75%         999.00000        331.000000
max        5645.00000        365.000000


In [10]:
# availability_365 < 0 — impossible given the column's meaning
neg_avail = df[df['availability_365'] < 0]
print("availability_365 < 0:", len(neg_avail))
print(neg_avail['room_type'].value_counts())
print(neg_avail['calculated_host_listings_count'].describe())


availability_365 < 0: 432
room_type
Entire home/apt    243
Private room       182
Shared room          7
Name: count, dtype: int64
count    429.000000
mean       5.016317
std       16.706432
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max      121.000000
Name: calculated_host_listings_count, dtype: float64


In [11]:
# availability_365 > 365 — the big one: 2.7% of the dataset, don't drop blindly
high_avail = df[df['availability_365'] > 365]
print("availability_365 > 365:", len(high_avail))

# .describe() profiles only numeric columns — tells you whether the
# outlier group is mildly over or wildly extreme
print(high_avail[['calculated_host_listings_count', 'availability_365', 'minimum_nights']].describe())

# value_counts() checks whether the anomaly clusters in one category
# (systemic bug) or spreads proportionally to the dataset (random noise)
print(high_avail['room_type'].value_counts())

# same test applied to hosts — repeated host ids would suggest bulk
# upload errors from a single source rather than scattered mistakes
print(high_avail['host_id'].value_counts().head(10))


availability_365 > 365: 2782
       calculated_host_listings_count  availability_365  minimum_nights
count                     2756.000000       2782.000000     2766.000000
mean                         4.542090        396.773904        7.114967
std                         15.363238         64.672218       16.737322
min                          1.000000        366.000000       -2.000000
25%                          1.000000        380.000000        1.000000
50%                          1.000000        395.000000        3.000000
75%                          2.000000        411.000000        5.000000
max                        121.000000       3677.000000      365.000000
room_type
Entire home/apt    1555
Private room       1184
Shared room          43
Name: count, dtype: int64
host_id
17669809925    2
59749079462    2
93968287670    2
37978937670    2
44533854808    2
87127860355    2
85335405136    2
83359846384    2
18401849962    2
94766740970    2
Name: count, dtype: int64


### Findings so far
- `minimum_nights` < 0, n=13 → negligible, drop outright.
- `minimum_nights` > 365, n=35 → mean 882 nights, max 5645 (15+ years
  as a "minimum stay") — clearly data entry errors. Drop, don't cap.
- `availability_365` < 0, n=432 → room-type split (Entire home/apt
  243, Private room 182, Shared room 7) roughly matches overall
  dataset proportions → scattered noise, not room-type-specific. Drop.
- `availability_365` > 365, n=2,782 (2.7% of dataset) → **still under
  investigation**. Host-id check pending confirmation: if top host
  ids repeat heavily, this points to bulk-upload errors from a few
  large hosts; if counts stay low (1-2 each), it's scattered
  individual errors with no single root cause. Decision on drop vs.
  cap vs. flag depends on this result — not yet locked.

## Next steps (not yet done)
1. Finish the `availability_365 > 365` host-id investigation, decide
   drop vs. cap vs. flag.
2. Clean `price` and `service_fee` — currently strings with a `$`
   prefix and trailing space, need conversion to numeric.
3. Drop `license` column entirely.
4. Decide treatment for `reviews_per_month` per the split above.
5. Frame final business questions once the dataset is verified clean.


In [12]:
# Store the row count before cleaning
rows_before = len(df)

# Drop rows where minimum_nights is negative
df = df[df['minimum_nights'] >= 0]

# Drop rows where minimum_nights is greater than 365
df = df[df['minimum_nights'] <= 365]

# Store the row count after cleaning
rows_after = len(df)

# Print the results
print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning: {rows_after}")
print(f"Rows dropped: {rows_before - rows_after}")

Rows before cleaning: 102599
Rows after cleaning: 102142
Rows dropped: 457


In [13]:
df = df[df['minimum_nights'] >= 0]
df = df[df['minimum_nights'] <= 365]

In [14]:
rows_before = len(df)

print(f"Rows before cleaning: {rows_before}")
df = df[df['minimum_nights'] >= 0]
print(f"Rows after cleaning: {len(df)}")
df = df[df['minimum_nights'] <= 365]

rows_after = len(df)

Rows before cleaning: 102142
Rows after cleaning: 102142


In [15]:
print(f"minimum_nights violations remaining: {((df['minimum_nights'] < 0) | (df['minimum_nights'] > 365)).sum()}")

minimum_nights violations remaining: 0


In [16]:
df.drop(columns=['license'], inplace=True)

In [17]:
print(df.head())

        id                                              name      host_id  \
0  1001254                Clean & quiet apt home by the park  80014485718   
1  1002102                             Skylit Midtown Castle  52335172823   
2  1002403               THE VILLAGE OF HARLEM....NEW YORK !  78829239556   
3  1002755                                               NaN  85098326012   
4  1003689  Entire Apt: Spacious Studio/Loft by central park  92037596077   

  host_identity_verified host_name neighbourhood_group neighbourhood  \
0            unconfirmed  Madaline            Brooklyn    Kensington   
1               verified     Jenna           Manhattan       Midtown   
2                    NaN     Elise           Manhattan        Harlem   
3            unconfirmed     Garry            Brooklyn  Clinton Hill   
4               verified    Lyndon           Manhattan   East Harlem   

        lat      long        country  ...  price service_fee minimum_nights  \
0  40.64749 -73.97237  Un

Conversion of $1,250 to   1250   as a float so by replacing $ and , with nothing and then converting string to a float

Doing the same for Service fee

In [18]:
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
df['service_fee'] = df['service_fee'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

In [19]:
##df['service_fee'] = df['service_fee'].str.replace('$','').str.replace(',','').astype(float)

In [20]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 102142 entries, 0 to 102598
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102142 non-null  int64  
 1   name                            101900 non-null  object 
 2   host_id                         102142 non-null  int64  
 3   host_identity_verified          101864 non-null  object 
 4   host_name                       101738 non-null  object 
 5   neighbourhood_group             102114 non-null  object 
 6   neighbourhood                   102127 non-null  object 
 7   lat                             102134 non-null  float64
 8   long                            102134 non-null  float64
 9   country                         101611 non-null  object 
 10  country_code                    102020 non-null  object 
 11  instant_bookable                102046 non-null  object 
 12  cancellation_policy  

In [21]:
print(df.isnull().sum())

id                                    0
name                                242
host_id                               0
host_identity_verified              278
host_name                           404
neighbourhood_group                  28
neighbourhood                        15
lat                                   8
long                                  8
country                             531
country_code                        122
instant_bookable                     96
cancellation_policy                  75
room_type                             0
construction_year                   202
price                               247
service_fee                         273
minimum_nights                        0
number_of_reviews                   181
last_review                       15812
reviews_per_month                 15804
review_rate_number                  305
calculated_host_listings_count      319
availability_365                    424
house_rules                       51920


In [22]:
# how many rows have reviews_per_month null AND number_of_reviews > 0?
mismatch = df[(df['reviews_per_month'].isnull()) & (df['number_of_reviews'] > 0)]
print(len(mismatch))

19


In [23]:
# Count missing values before cleaning
print("Missing before:", df['reviews_per_month'].isnull().sum())

# Step 1: Fill missing reviews_per_month with 0 ONLY for listings with 0 reviews
df.loc[
    (df['number_of_reviews'] == 0) &
    (df['reviews_per_month'].isna()),
    'reviews_per_month'
] = 0

# Step 2: Check how many missing values remain
print("Missing after:", df['reviews_per_month'].isnull().sum())

# Step 3: Display the remaining rows with missing reviews_per_month
remaining_missing = df[df['reviews_per_month'].isna()]

print("\nRemaining rows with missing reviews_per_month:")
print(remaining_missing[
    ['id', 'host_id', 'name', 'number_of_reviews', 'reviews_per_month']
])

Missing before: 15804
Missing after: 142

Remaining rows with missing reviews_per_month:
             id      host_id  \
163     1091361  86944769515   
164     1091913  20270952150   
165     1092466   3310140241   
166     1093018  61571782497   
168     1094122  45745264571   
...         ...          ...   
91430  51498125  79596009989   
91431  51498677   2317212823   
91432  51499230  47335669596   
98718  55523287  71618038009   
98719  55523840  25233846702   

                                                    name  number_of_reviews  \
163                     Private, Large & Sunny 1BR w/W&D              309.0   
164                   Luxurious Condo in DUBMO with View               14.0   
165      Charming & Cozy midtown loft any WEEK ENDS  !!!                4.0   
166    * Spacious GARDEN Park Slope Duplex* 6 people max               80.0   
168                   Parlor Room In Victorian Townhouse              294.0   
...                                                 

In [24]:
print((remaining_missing['number_of_reviews'] > 0).sum())
print((remaining_missing['number_of_reviews'] == 0).sum())

19
0


In [25]:
print(remaining_missing['number_of_reviews'].isnull().sum())
print(remaining_missing['number_of_reviews'].describe())

123
count     19.000000
mean     102.526316
std       96.264894
min        2.000000
25%       15.500000
50%       80.000000
75%      162.000000
max      309.000000
Name: number_of_reviews, dtype: float64


In [26]:
# Run this as ONE block. Do not run other cells before or after until you've read the output.

print("Total rows in df:", len(df))
print("Total nulls in reviews_per_month:", df['reviews_per_month'].isnull().sum())
print("Total nulls in number_of_reviews:", df['number_of_reviews'].isnull().sum())

mismatch = df[(df['reviews_per_month'].isnull()) & (df['number_of_reviews'] > 0)]
print("\nMismatch (reviews_per_month null, number_of_reviews > 0):", len(mismatch))

still_zero = df[(df['reviews_per_month'].isnull()) & (df['number_of_reviews'] == 0)]
print("Still-zero unfilled (should be 0 if Step 1 worked):", len(still_zero))

nan_reviews = df[(df['reviews_per_month'].isnull()) & (df['number_of_reviews'].isnull())]
print("Both null (edge case, neither condition caught these):", len(nan_reviews))

total_remaining = df['reviews_per_month'].isnull().sum()
accounted_for = len(mismatch) + len(still_zero) + len(nan_reviews)
print(f"\nSanity check — total remaining nulls: {total_remaining}, accounted for: {accounted_for}")
print("These two numbers MUST match. If they don't, something in df changed between definitions.")

Total rows in df: 102142
Total nulls in reviews_per_month: 142
Total nulls in number_of_reviews: 181

Mismatch (reviews_per_month null, number_of_reviews > 0): 19
Still-zero unfilled (should be 0 if Step 1 worked): 0
Both null (edge case, neither condition caught these): 123

Sanity check — total remaining nulls: 142, accounted for: 142
These two numbers MUST match. If they don't, something in df changed between definitions.


In [27]:
output_path = r"C:\Users\yashv\Downloads\airbnb_cleaned.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows, {df.shape[1]} columns to {output_path}")

Saved 102142 rows, 25 columns to C:\Users\yashv\Downloads\airbnb_cleaned.csv


In [28]:
df[df['country'].isnull()][['neighbourhood_group','lat','long']].describe()

,lat,long
count,531.000000,531.000000
mean,40.726300,-73.952842
std,0.055897,0.047040
min,40.559660,-74.156800
25%,40.689500,-73.982975
50%,40.723110,-73.955790
75%,40.762205,-73.934240
max,40.901110,-73.755780


In [29]:
df[['price','service_fee']].corr()

,price,service_fee
price,1.000000,0.999991
service_fee,0.999991,1.000000


In [30]:
df['review_rate_number'].value_counts()

review_rate_number
5.0    23283
4.0    23246
3.0    23156
2.0    22993
1.0     9159
Name: count, dtype: int64

In [31]:
df['service_fee'] = df['service_fee'].fillna(df['price'] * 0.20)

In [32]:
print(df['service_fee'].isnull().sum())
print(df[df['service_fee'].isnull()]['price'].isnull().sum())

34
34


In [33]:
df['country'] = df['country'].fillna('United States')
df['country_code'] = df['country_code'].fillna('US')

In [34]:
df['review_rate_number'] = df['review_rate_number'].fillna(df['review_rate_number'].median())

In [35]:
print(df['calculated_host_listings_count'].describe())
print(df['calculated_host_listings_count'].isnull().sum())

count    101823.000000
mean          7.934524
std          32.194386
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max         332.000000
Name: calculated_host_listings_count, dtype: float64
319


In [36]:
df['calculated_host_listings_count'] = df['calculated_host_listings_count'].fillna(df['calculated_host_listings_count'].median())

In [37]:
df = df.dropna(subset=['price', 'service_fee'])

In [38]:
print(df.isnull().sum())

id                                    0
name                                239
host_id                               0
host_identity_verified              274
host_name                           399
neighbourhood_group                  27
neighbourhood                        14
lat                                   8
long                                  8
country                               0
country_code                          0
instant_bookable                     91
cancellation_policy                  70
room_type                             0
construction_year                   198
price                                 0
service_fee                           0
minimum_nights                        0
number_of_reviews                   181
last_review                       15786
reviews_per_month                   141
review_rate_number                    0
calculated_host_listings_count        0
availability_365                    424
house_rules                       51785


In [39]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 101895 entries, 0 to 102598
Data columns (total 25 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              101895 non-null  int64  
 1   name                            101656 non-null  object 
 2   host_id                         101895 non-null  int64  
 3   host_identity_verified          101621 non-null  object 
 4   host_name                       101496 non-null  object 
 5   neighbourhood_group             101868 non-null  object 
 6   neighbourhood                   101881 non-null  object 
 7   lat                             101887 non-null  float64
 8   long                            101887 non-null  float64
 9   country                         101895 non-null  object 
 10  country_code                    101895 non-null  object 
 11  instant_bookable                101804 non-null  object 
 12  cancellation_policy  

In [40]:
print(df.head())

        id                                              name      host_id  \
0  1001254                Clean & quiet apt home by the park  80014485718   
1  1002102                             Skylit Midtown Castle  52335172823   
2  1002403               THE VILLAGE OF HARLEM....NEW YORK !  78829239556   
3  1002755                                               NaN  85098326012   
4  1003689  Entire Apt: Spacious Studio/Loft by central park  92037596077   

  host_identity_verified host_name neighbourhood_group neighbourhood  \
0            unconfirmed  Madaline            Brooklyn    Kensington   
1               verified     Jenna           Manhattan       Midtown   
2                    NaN     Elise           Manhattan        Harlem   
3            unconfirmed     Garry            Brooklyn  Clinton Hill   
4               verified    Lyndon           Manhattan   East Harlem   

        lat      long        country  ...  price service_fee minimum_nights  \
0  40.64749 -73.97237  Un

In [41]:
print(df['availability_365'].describe())

count    101471.000000
mean        140.992165
std         135.431574
min         -10.000000
25%           3.000000
50%          96.000000
75%         268.000000
max        3677.000000
Name: availability_365, dtype: float64


In [42]:
# Step 1: cap mild overshoot (366-400) at 365 — same threshold we justified earlier
df.loc[(df['availability_365'] >= 366) & (df['availability_365'] <= 400), 'availability_365'] = 365

# Step 2: drop severe overshoot (>400) — not salvageable by capping
df = df[(df['availability_365'] <= 400) | (df['availability_365'].isnull())]

# Step 3: drop negative values — physically impossible
df = df[(df['availability_365'] >= 0) | (df['availability_365'].isnull())]

# Verify
print(df['availability_365'].describe())

count    99918.000000
mean       138.209942
std        132.002206
min          0.000000
25%          3.000000
50%         94.000000
75%        263.000000
max        365.000000
Name: availability_365, dtype: float64


In [43]:
print(df['availability_365'].describe())

count    99918.000000
mean       138.209942
std        132.002206
min          0.000000
25%          3.000000
50%         94.000000
75%        263.000000
max        365.000000
Name: availability_365, dtype: float64


In [44]:
print(df.isnull().sum())

id                                    0
name                                233
host_id                               0
host_identity_verified              269
host_name                           398
neighbourhood_group                  27
neighbourhood                        12
lat                                   8
long                                  8
country                               0
country_code                          0
instant_bookable                     85
cancellation_policy                  65
room_type                             0
construction_year                   186
price                                 0
service_fee                           0
minimum_nights                        0
number_of_reviews                   180
last_review                       15506
reviews_per_month                   140
review_rate_number                    0
calculated_host_listings_count        0
availability_365                    424
house_rules                       51285
